# 3. Machine Learning Models

**Pipeline position:** runs after `1_data_prep.ipynb`.

## What this notebook does

Fits four supervised models on the forest-adjusted panel and saves their
outputs in shapes the viz notebooks can plot directly:

| Model | Purpose | Used by chart |
|---|---|---|
| LASSO (with cross-validated alpha) | sparse coefficient estimate | 7 |
| Ridge (cross-validated alpha) | dense coefficient estimate | 7 |
| Elastic Net (l1_ratio = 0.5) | middle-ground coefficient estimate | 7 |
| Random Forest (200 trees, depth 4) | non-parametric feature importance | 8 |

After fitting on the train window (1995–2014), the same models are refit on
the full panel and used to roll a 2020–2030 forecast forward year-by-year
for each country (chart 9). A 200-iteration country-block bootstrap of the
LASSO coefficients is also produced for chart 21.

## Why this is one notebook, not three

Chart 7, 8, and 9 all need the same fitted models. Putting them in three
separate notebooks would force three identical fits. Charting from cached
artifacts means every viz notebook is fast.

## Outputs

- `ml_coefficients.csv` — feature, LASSO coef, Ridge coef, EN coef, RF MDI
  importance, mean absolute coefficient, normalised RF importance
- `ml_forecast.csv` — country–year ensemble forecast 2020–2030 with
  per-model predictions and standard deviation across models
- `ml_forecast_history.csv` — historical ECI (1995–2019) for the same
  countries, used to draw the solid line before the dashed forecast
- `lasso_bootstrap.csv` — 200 × n_features matrix of LASSO coefficients
  across country-block bootstrap iterations


In [1]:
import os, sys, pickle
sys.path.insert(0, '.')
import numpy as np
import pandas as pd

from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.ensemble    import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

from _config import (INTERMEDIARY, ARTIFACTS, ECI_COL, BASE_FEATS, INTERACTION_FEATS,
                     ALL_FEATS, NAME_MAP, artifact_path)

panel = pd.read_parquet(artifact_path('panel.parquet'))
print(f'Panel: {panel["Country Code"].nunique()} countries, {len(panel):,} obs')


Panel: 107 countries, 2,675 obs


## Build the ML dataset

Drop the first year per country (no L1_ECI), drop rows with missing values
in any model feature.


In [2]:
ml_df = panel.dropna(subset=['L1_ECI', ECI_COL]).copy()
ml_df = ml_df.dropna(subset=ALL_FEATS)
print(f'ML sample: {ml_df["Country Code"].nunique()} countries, {len(ml_df):,} obs')


ML sample: 104 countries, 2,466 obs


## Cross-validation splitter

Expanding-window panel CV with a one-year gap, mirroring the V5 setup so
results are comparable. The class is small enough to live in this notebook.


In [3]:
class PanelTemporalCV:
    def __init__(self, years, n_splits=5, gap=1, min_train_years=8):
        uy = np.sort(np.unique(years))
        ec = uy[0]  + min_train_years - 1
        lc = uy[-1] - gap - 1
        self.cutoffs  = np.unique(np.linspace(ec, lc, n_splits).astype(int))
        self.n_splits = len(self.cutoffs)
        self.years    = np.asarray(years)
        self.gap      = gap

    def split(self, X=None, y=None, groups=None):
        for c in self.cutoffs:
            ti = np.where(self.years <= c)[0]
            vi = np.where(self.years > c + self.gap)[0]
            if len(ti) and len(vi):
                yield ti, vi

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


## Fit on the train window (1995–2014)

These coefficients feed charts 7 and 8. The cache is invalidated whenever
the sample size changes, so re-running notebook 1 with a different
threshold automatically forces a refit here.


In [4]:
train_df = ml_df[ml_df['Year'] <= 2014].copy()
test_df  = ml_df[ml_df['Year'] >= 2015].copy()

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[ALL_FEATS].values)
X_test  = scaler.transform(test_df[ALL_FEATS].values)
y_train = train_df[ECI_COL].values
y_test  = test_df[ECI_COL].values

tscv = PanelTemporalCV(train_df['Year'].values, n_splits=5, gap=1, min_train_years=8)

CACHE = artifact_path('ml_cache.pkl')
SKIP_FIT = os.environ.get('SKIP_ML_FIT') == '1'

if SKIP_FIT and os.path.exists(CACHE):
    with open(CACHE, 'rb') as f:
        cache = pickle.load(f)
    lasso, ridge, elastic, rf = cache['lasso'], cache['ridge'], cache['elastic'], cache['rf']
    print('  Loaded train-window models from cache')
elif os.path.exists(CACHE):
    with open(CACHE, 'rb') as f:
        cache = pickle.load(f)
    if cache.get('n_countries') == ml_df['Country Code'].nunique():
        lasso, ridge, elastic, rf = cache['lasso'], cache['ridge'], cache['elastic'], cache['rf']
        print('  Cache valid, loaded train-window models')
    else:
        cache = None

if 'cache' not in dir() or cache is None or 'lasso' not in dir():
    print('  Fitting LASSO …')
    lasso   = LassoCV(cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_train)
    print('  Fitting Ridge …')
    ridge   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv).fit(X_train, y_train)
    print('  Fitting Elastic Net …')
    elastic = ElasticNetCV(l1_ratio=[0.5], cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_train)
    print('  Fitting Random Forest …')
    rf      = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                    random_state=42, n_jobs=-1).fit(X_train, y_train)
    cache = dict(lasso=lasso, ridge=ridge, elastic=elastic, rf=rf,
                 n_countries=ml_df['Country Code'].nunique())


  Fitting LASSO …
  Fitting Ridge …
  Fitting Elastic Net …
  Fitting Random Forest …


## Save coefficients and importances

`ml_coefficients.csv` is the only artifact charts 7 and 8 need. The viz
notebook truncates to top 12 by `abs_avg` for chart 7 and to top 12 by
`RF_norm` for chart 8.


In [5]:
def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo + 1e-12)

short = [NAME_MAP.get(f, f) for f in ALL_FEATS]
coef_df = pd.DataFrame({
    'feature':  ALL_FEATS,
    'short':    short,
    'lasso':    lasso.coef_,
    'ridge':    ridge.coef_,
    'en':       elastic.coef_,
    'rf':       rf.feature_importances_,
})
coef_df['abs_avg'] = coef_df[['lasso', 'ridge', 'en']].abs().mean(axis=1)
coef_df['rf_norm'] = minmax(coef_df['rf'].values)
coef_df.to_csv(artifact_path('ml_coefficients.csv'), index=False)
print(f'  Saved ml_coefficients.csv ({len(coef_df)} features)')


  Saved ml_coefficients.csv (21 features)


## Refit on the full panel for forecasting

The train-window models are biased toward the end of the train period.
Forecasting 2020–2030 needs models fit on the entire 1995–2019 panel.


In [6]:
X_full_raw = ml_df[ALL_FEATS].values
y_full     = ml_df[ECI_COL].values
scaler_full = StandardScaler()
X_full = scaler_full.fit_transform(X_full_raw)
tscv_full = PanelTemporalCV(ml_df['Year'].values, n_splits=5, gap=1, min_train_years=8)

if 'fc_lasso' not in cache:
    print('  Fitting full-sample LASSO …')
    fc_lasso   = LassoCV(cv=tscv_full, random_state=42, max_iter=10000).fit(X_full, y_full)
    print('  Fitting full-sample Ridge …')
    fc_ridge   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv_full).fit(X_full, y_full)
    print('  Fitting full-sample Elastic Net …')
    fc_elastic = ElasticNetCV(l1_ratio=[0.5], cv=tscv_full, random_state=42, max_iter=10000).fit(X_full, y_full)
    print('  Fitting full-sample Random Forest …')
    fc_rf      = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                       random_state=42, n_jobs=-1).fit(X_full, y_full)
    cache.update(dict(fc_lasso=fc_lasso, fc_ridge=fc_ridge, fc_elastic=fc_elastic,
                      fc_rf=fc_rf, scaler_full=scaler_full))
    with open(CACHE, 'wb') as f:
        pickle.dump(cache, f)
    print('  Cached.')
else:
    fc_lasso   = cache['fc_lasso']
    fc_ridge   = cache['fc_ridge']
    fc_elastic = cache['fc_elastic']
    fc_rf      = cache['fc_rf']
    scaler_full = cache['scaler_full']
    print('  Loaded full-sample models from cache')


  Fitting full-sample LASSO …
  Fitting full-sample Ridge …
  Fitting full-sample Elastic Net …
  Fitting full-sample Random Forest …
  Cached.


## Build the 2020–2030 forecast

For each country, project each feature forward year by year using a linear
trend on the last 5 observed years, feed the projected feature vector into
each model, and use the predicted ECI as the next year's `L1_ECI`. The
ensemble is the simple mean across the four models; the standard deviation
across models is the forecast uncertainty.


In [7]:
FORECAST_YEARS = list(range(2020, 2031))
trend_feats = [f for f in BASE_FEATS if f != 'L1_ECI']

# Mean-centring constants for re-computing interactions on projected features.
hci_m  = ml_df['Human capital index'].mean()
gfcf_m = ml_df['Gross fixed capital formation, all, Constant prices, Percent of GDP'].mean()
prod_m = ml_df['Total_Production_Value_Per_Capita'].mean()


def extrapolate_country(cdf, yrs):
    last5 = cdf.tail(5)
    rows = []
    for yr in yrs:
        row = {
            'Year':         yr,
            'Country Code': cdf['Country Code'].iloc[0],
            'Country Name': cdf['Country Name'].iloc[0],
        }
        for feat in trend_feats:
            vals = last5[feat].dropna().values
            if len(vals) >= 2:
                slope, intercept = np.polyfit(np.arange(len(vals)), vals, 1)
                steps = yr - int(last5['Year'].iloc[-1])
                row[feat] = float(intercept + slope * (len(vals) - 1 + steps))
            else:
                row[feat] = float(vals[-1]) if len(vals) else 0.0
        rows.append(row)
    return rows


future_rows = []
for cc, cdf in ml_df.groupby('Country Code'):
    future_rows.extend(extrapolate_country(cdf, FORECAST_YEARS))
future_df = pd.DataFrame(future_rows)


fc_models = {'LASSO': fc_lasso, 'Ridge': fc_ridge,
             'Elastic Net': fc_elastic, 'RF': fc_rf}
records = []
for cc, cdf in ml_df.groupby('Country Code'):
    last_eci = float(cdf.sort_values('Year')[ECI_COL].iloc[-1])
    fsub     = future_df[future_df['Country Code'] == cc].sort_values('Year')
    running  = {n: last_eci for n in fc_models}
    for _, frow in fsub.iterrows():
        preds = {}
        for name, model in fc_models.items():
            row2 = frow.copy()
            row2['L1_ECI'] = running[name]
            row2['HCI_x_ProductionValue']  = (
                (row2['Human capital index'] - hci_m)
                * (row2['Total_Production_Value_Per_Capita'] - prod_m)
            )
            row2['GFCF_x_ProductionValue'] = (
                (row2['Gross fixed capital formation, all, Constant prices, Percent of GDP'] - gfcf_m)
                * (row2['Total_Production_Value_Per_Capita'] - prod_m)
            )
            x_vec = np.array([row2.get(f, 0) for f in ALL_FEATS]).reshape(1, -1)
            pred  = model.predict(scaler_full.transform(x_vec))[0]
            preds[name]   = pred
            running[name] = pred
        rec = {
            'Country Code':   cc,
            'Country Name':   frow['Country Name'],
            'Year':           int(frow['Year']),
            'Last_Known_ECI': last_eci,
            'Ensemble':       float(np.mean(list(preds.values()))),
            'ECI_std':        float(np.std(list(preds.values()))),
        }
        rec.update(preds)
        records.append(rec)

forecast_df = pd.DataFrame(records)
forecast_df.to_csv(artifact_path('ml_forecast.csv'), index=False)
print(f'  Saved ml_forecast.csv ({len(forecast_df):,} rows)')


  Saved ml_forecast.csv (1,144 rows)


## Save historical ECI for the same countries

Chart 9 needs the solid (historical) part of each country's line. Save it
once here so the viz notebook does no re-loading.


In [8]:
master_full = pd.read_csv(os.path.join(INTERMEDIARY, 'Master.csv'))
hist = (master_full[master_full['Country Code'].isin(panel['Country Code'].unique())]
        [['Country Code', 'Country Name', 'Year', ECI_COL]].dropna())
hist.to_csv(artifact_path('ml_forecast_history.csv'), index=False)
print(f'  Saved ml_forecast_history.csv ({len(hist):,} rows)')


  Saved ml_forecast_history.csv (2,675 rows)


## LASSO bootstrap (chart 21)

Country-block bootstrap with B=200 iterations. Each iteration samples 38
countries with replacement and refits LASSO on their pooled time series.
This takes a few minutes; set `SKIP_BOOTSTRAP=1` in the environment to skip.


In [9]:
SKIP_BOOTSTRAP = os.environ.get('SKIP_BOOTSTRAP') == '1'
BOOT_PATH = artifact_path('lasso_bootstrap.csv')

if SKIP_BOOTSTRAP and os.path.exists(BOOT_PATH):
    print(f'  Skipping bootstrap; using existing {os.path.basename(BOOT_PATH)}')
else:
    B = int(os.environ.get('BOOTSTRAP_B', 200))
    rng = np.random.default_rng(42)
    countries = ml_df['Country Code'].unique()
    n = len(countries)
    boot_rows = []
    for b in range(B):
        if b % 25 == 0:
            print(f'    iter {b}/{B}')
        sample_cc = rng.choice(countries, size=n, replace=True)
        # Concat each country (replicated as drawn) keeping order.
        parts = []
        for cc in sample_cc:
            parts.append(ml_df[ml_df['Country Code'] == cc])
        sub = pd.concat(parts, ignore_index=True)
        sub = sub[sub['Year'] <= 2014]
        if len(sub) < 50:
            continue
        sc_b = StandardScaler()
        Xb = sc_b.fit_transform(sub[ALL_FEATS].values)
        yb = sub[ECI_COL].values
        tscv_b = PanelTemporalCV(sub['Year'].values, n_splits=5, gap=1, min_train_years=8)
        try:
            las_b = LassoCV(cv=tscv_b, random_state=42 + b, max_iter=5000).fit(Xb, yb)
        except Exception:
            continue
        for fi, f in enumerate(ALL_FEATS):
            boot_rows.append(dict(iter=b, feature=f,
                                  short=NAME_MAP.get(f, f),
                                  coef=float(las_b.coef_[fi])))
    boot_df = pd.DataFrame(boot_rows)
    boot_df.to_csv(BOOT_PATH, index=False)
    print(f'  Saved lasso_bootstrap.csv ({len(boot_df):,} rows, '
          f'{boot_df["iter"].nunique()} iterations)')


    iter 0/200
    iter 25/200
    iter 50/200
    iter 75/200
    iter 100/200
    iter 125/200
    iter 150/200
    iter 175/200
  Saved lasso_bootstrap.csv (4,200 rows, 200 iterations)


## Summary

| Artifact | Rows | Used by |
|---|---|---|
| ml_coefficients.csv | one per feature | charts 7, 8 |
| ml_forecast.csv | country × forecast year | chart 9 |
| ml_forecast_history.csv | country × historical year | chart 9 |
| lasso_bootstrap.csv | 200 × n_features | chart 21 |

Models also cached at `artifacts/ml_cache.pkl`. To skip refitting:
`SKIP_ML_FIT=1 jupyter nbconvert --execute …`
